# ACS 2022 (5-Year) Preprocessing

This notebook preprocesses **ACS 2022 5-Year** county-level tables and produces a clean, merge-ready feature dataset keyed by a standardized **5-digit county FIPS code**.  
The resulting dataset serves as the primary socioeconomic feature matrix to be merged with Zillow county-level housing prices.


## Processed Tables and Outputs

The following ACS tables are processed:

- **B19013 (Income)** → `median_income_2022`  
- **B01003 (Population)** → `population_2022`  
- **B15003 (Education)** →  
  - `ba_plus_share_2022` (Bachelor’s degree or higher)  
  - `hs_plus_share_2022` (High school diploma or higher)  
- **B25010 (Household size)** → `avg_household_size_2022`  
- **B23025 (Employment)** →  
  - `unemployment_rate_2022`  
  - Supporting counts: `civilian_labor_force_2022`, `employed_2022`, `unemployed_2022`

Margins of error are not used, as the goal is interpretability rather than survey uncertainty.  
Puerto Rico counties (state FIPS = 72) are excluded to ensure consistency with Zillow county coverage.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd().parent
RAW_CENSUS_DIR = BASE_DIR / "data" / "raw" / "census"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ACS_TABLES = {
    "income": {
        "file": "B19013_income.csv",
        "output": "acs_income_b19013_2022.csv",
    },
    "population": {
        "file": "B01003_population.csv",
        "output": "acs_population_b01003_2022.csv",
    },
    "education": {
        "file": "B15003_education.csv",
        "output": "acs_education_b15003_2022.csv",
    },
    "household": {
        "file": "B25010_household_size.csv",
        "output": "acs_household_b25010_2022.csv",
    },
    "employment": {
        "file": "B23025_employment.csv",
        "output": "acs_employment_b23025_2022.csv",
    },
}

# Exclude Puerto Rico (state FIPS = 72)
EXCLUDE_STATE_FIPS = {"72"}


## Common Preprocessing Principles

All ACS tables follow the same preprocessing rules:

- **County-level alignment** using standardized 5-digit county FIPS codes extracted from `GEO_ID`
- **Non-county rows removed** (e.g., national or state totals)
- **Puerto Rico excluded** (FIPS starting with 72)
- **Safe numeric conversion** (`errors="coerce"`) to handle suppressed ACS entries
- **Explicit data quality checks**, including:
  - Valid ranges for rates and shares
  - Positive population and labor force counts
- **Defensive guardrails** to prevent silent geographic mismatches

These steps ensure consistent geographic alignment and reliable downstream merging.


In [2]:
def load_acs_table(key: str) -> pd.DataFrame:
    cfg = ACS_TABLES[key]
    input_path = RAW_CENSUS_DIR / cfg["file"]
    df = pd.read_csv(input_path)

    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")].copy()
    return df

def drop_excluded_state_fips(df: pd.DataFrame, fips_col: str = "fips") -> pd.DataFrame:
    """
    Drop rows whose state FIPS (first 2 digits of 5-digit county FIPS) is excluded.
    """
    out = df.copy()
    out[fips_col] = out[fips_col].astype(str).str.zfill(5)
    state_fips = out[fips_col].str[:2]
    out = out[~state_fips.isin(EXCLUDE_STATE_FIPS)].copy()
    return out

def standardize_county_fips(df: pd.DataFrame, geo_col: str = "GEO_ID") -> pd.DataFrame:
    """
    Extract the last 5 digits from GEO_ID as county FIPS and keep only valid county rows.
    Excludes Puerto Rico (state_fips=72).
    """
    out = df.copy()

    # Extract last 5 chars as FIPS
    out["fips"] = out[geo_col].astype(str).str[-5:]

    # Keep only proper 5-digit numeric county FIPS rows
    out = out[out["fips"].str.fullmatch(r"\d{5}")].copy()

    # Exclude Puerto Rico (72xxx)
    out = drop_excluded_state_fips(out, fips_col="fips")

    return out

def to_numeric_safe(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def assert_no_pr(df: pd.DataFrame, name: str) -> None:
    """
    Debug guard: ensure no Puerto Rico rows remain.
    """
    n = df["fips"].astype(str).str.zfill(5).str.startswith("72").sum()
    if n > 0:
        raise ValueError(f"[{name}] Puerto Rico rows still present: {n}")

def save_processed(df: pd.DataFrame, key: str) -> None:
    output_name = ACS_TABLES[key]["output"]
    output_path = PROCESSED_DIR / output_name
    df.to_csv(output_path, index=False)
    print(f"Saved: data/processed/{output_name}")
    

## Income (B19013)

Median household income is derived from **B19013_001E** and stored as `median_income_2022`, expressed in inflation-adjusted 2022 dollars.

- Suppressed values (`-`) are converted to missing and removed
- Counties without a valid income estimate are excluded

**Note:** One county (Loving County, TX; FIPS 48301) is excluded due to a suppressed income estimate.

**Output columns:**
- `fips`
- `county_name`
- `median_income_2022`


In [3]:
def preprocess_income(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_county_fips(df)

    keep = ["fips", "NAME", "B19013_001E"]
    out = df[keep].copy()

    # explicit missing token handling
    out["B19013_001E"] = out["B19013_001E"].replace({"-": np.nan})

    out = to_numeric_safe(out, ["B19013_001E"])
    out = out.rename(columns={
        "NAME": "county_name",
        "B19013_001E": "median_income_2022",
    })

    out = out.dropna(subset=["median_income_2022"]).copy()
    return out

income_raw = load_acs_table("income")
income = preprocess_income(income_raw)
assert_no_pr(income, "income")

income.shape, income.head()


((3143, 3),
     fips              county_name  median_income_2022
 1  01001  Autauga County, Alabama             68315.0
 2  01003  Baldwin County, Alabama             71039.0
 3  01005  Barbour County, Alabama             39712.0
 4  01007     Bibb County, Alabama             50669.0
 5  01009   Blount County, Alabama             57440.0)

In [4]:
income_out = PROCESSED_DIR / ACS_TABLES["income"]["output"]
income.to_csv(income_out, index=False)
save_processed(income, "income")


Saved: data/processed/acs_income_b19013_2022.csv


## Population (B01003)

Total county population is taken from **B01003_001E** and stored as `population_2022`.

- Non-numeric or missing values are removed
- Counties with non-positive population counts are excluded

**Output columns:**
- `fips`
- `county_name`
- `population_2022`


In [5]:
def preprocess_population(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_county_fips(df)

    keep = ["fips", "NAME", "B01003_001E"]
    out = df[keep].copy()
    out = to_numeric_safe(out, ["B01003_001E"])

    out = out.rename(columns={
        "NAME": "county_name",
        "B01003_001E": "population_2022",
    })

    out = out.dropna(subset=["population_2022"]).copy()
    out = out[out["population_2022"] > 0].copy()
    return out

pop_raw = load_acs_table("population")
population = preprocess_population(pop_raw)
assert_no_pr(population, "population")

population.shape, population.head()


((3144, 3),
     fips              county_name  population_2022
 1  01001  Autauga County, Alabama            58761
 2  01003  Baldwin County, Alabama           233420
 3  01005  Barbour County, Alabama            24877
 4  01007     Bibb County, Alabama            22251
 5  01009   Blount County, Alabama            59077)

In [6]:
pop_out = PROCESSED_DIR / ACS_TABLES["population"]["output"]
population.to_csv(pop_out, index=False)
save_processed(population, "population")


Saved: data/processed/acs_population_b01003_2022.csv


## Cross-table Consistency Check (Income vs Population)

As a validation step, county FIPS codes in the income and population tables are compared.

- Exactly one county appears in population but not in income (FIPS 48301)
- Inspection confirms this discrepancy is due to a suppressed income value, not a preprocessing error

This check verifies that row count differences are expected and data-driven.


In [7]:
income_fips = set(income["fips"])
pop_fips = set(population["fips"])

missing_in_income = sorted(pop_fips - income_fips)
missing_in_pop = sorted(income_fips - pop_fips)

len(missing_in_income), missing_in_income[:10], len(missing_in_pop), missing_in_pop[:10]


(1, ['48301'], 0, [])

In [8]:
target = missing_in_income[0]
income_raw[income_raw["GEO_ID"].astype(str).str.endswith(target)].head(5)


,GEO_ID,NAME,B19013_001E,B19013_001M
2676,0500000US48301,"Loving County, Texas",-,**


## Education (B15003)

Educational attainment is summarized into two interpretable shares based on **B15003**, whose universe is the adult population (typically age 25+):

- **`ba_plus_share_2022`**  
  Share of adults with a Bachelor’s degree or higher  

- **`hs_plus_share_2022`**  
  Share of adults with a high school diploma (or equivalent) or higher  

Quality constraints:
- Shares must lie in [0, 1]
- `hs_plus_share_2022 ≥ ba_plus_share_2022`

These constraints enforce internal consistency and remove invalid observations.

**Output columns:**
- `fips`
- `county_name`
- `ba_plus_share_2022`
- `hs_plus_share_2022`


In [9]:
def preprocess_education(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build county-level education features from ACS B15003.

    Outputs:
    - ba_plus_share_2022 = (Bachelor + Master + Professional + Doctorate) / Total
    - hs_plus_share_2022 = (High school diploma or higher) / Total
    """
    df = standardize_county_fips(df)

    total_col = "B15003_001E"

    ba_plus_cols = ["B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E"]
    hs_plus_cols = [f"B15003_{i:03d}E" for i in range(17, 26)]  # 017E..025E

    keep = ["fips", "NAME", total_col] + hs_plus_cols
    keep = list(dict.fromkeys(keep))

    out = df[keep].copy()

    numeric_cols = [total_col] + hs_plus_cols
    numeric_cols = list(dict.fromkeys(numeric_cols))
    out = to_numeric_safe(out, numeric_cols)

    out = out.dropna(subset=[total_col]).copy()
    out = out[out[total_col] > 0].copy()

    out["ba_plus_share_2022"] = out[ba_plus_cols].sum(axis=1) / out[total_col]
    out["hs_plus_share_2022"] = out[hs_plus_cols].sum(axis=1) / out[total_col]

    out = out.rename(columns={"NAME": "county_name"})
    out = out[["fips", "county_name", "ba_plus_share_2022", "hs_plus_share_2022"]].copy()

    out = out[
        (out["ba_plus_share_2022"].between(0, 1))
        & (out["hs_plus_share_2022"].between(0, 1))
        & (out["hs_plus_share_2022"] >= out["ba_plus_share_2022"])
    ].copy()

    return out

edu_raw = load_acs_table("education")
education = preprocess_education(edu_raw)
assert_no_pr(education, "education")

education.shape, education.head()


((3144, 4),
     fips              county_name  ba_plus_share_2022  hs_plus_share_2022
 1  01001  Autauga County, Alabama            0.295586            0.904026
 2  01003  Baldwin County, Alabama            0.325616            0.915993
 3  01005  Barbour County, Alabama            0.118812            0.764922
 4  01007     Bibb County, Alabama            0.109199            0.788634
 5  01009   Blount County, Alabama            0.147414            0.817552)

In [10]:
edu_out = PROCESSED_DIR / ACS_TABLES["education"]["output"]
education.to_csv(edu_out, index=False)
save_processed(education, "education")


Saved: data/processed/acs_education_b15003_2022.csv


## Household Size (B25010)

Average household size is taken from **B25010_001E** and stored as `avg_household_size_2022`.

- Values are converted to numeric
- Non-positive values are excluded

Owner-occupied and renter-occupied splits are supported by the preprocessing function but omitted here to keep the feature set compact and interpretable.

**Output columns:**
- `fips`
- `county_name`
- `avg_household_size_2022`


In [11]:
def preprocess_household(df: pd.DataFrame, include_owner_renter: bool = False) -> pd.DataFrame:
    df = standardize_county_fips(df)

    keep = ["fips", "NAME", "B25010_001E"]
    rename = {
        "NAME": "county_name",
        "B25010_001E": "avg_household_size_2022",
    }

    if include_owner_renter:
        keep += ["B25010_002E", "B25010_003E"]
        rename.update({
            "B25010_002E": "avg_hh_size_owner_2022",
            "B25010_003E": "avg_hh_size_renter_2022",
        })

    out = df[keep].copy()
    out = to_numeric_safe(out, [c for c in keep if c.startswith("B25010_")])

    out = out.dropna(subset=["B25010_001E"]).copy()
    out = out.rename(columns=rename)

    out = out[out["avg_household_size_2022"] > 0].copy()
    return out

hh_raw = load_acs_table("household")
household = preprocess_household(hh_raw, include_owner_renter=False)
assert_no_pr(household, "household")

household.shape, household.head()


((3144, 3),
     fips              county_name  avg_household_size_2022
 1  01001  Autauga County, Alabama                     2.61
 2  01003  Baldwin County, Alabama                     2.53
 3  01005  Barbour County, Alabama                     2.43
 4  01007     Bibb County, Alabama                     2.89
 5  01009   Blount County, Alabama                     2.70)

In [12]:
hh_out = PROCESSED_DIR / ACS_TABLES["household"]["output"]
household.to_csv(hh_out, index=False)
save_processed(household, "household")


Saved: data/processed/acs_household_b25010_2022.csv


## Employment (B23025)

Labor market characteristics are derived from **B23025**, which covers the civilian labor force (age 16+):

- `civilian_labor_force_2022` (B23025_003E)
- `employed_2022` (B23025_004E)
- `unemployed_2022` (B23025_005E)

An interpretable unemployment rate is constructed as:

$$
\text{unemployment\_rate\_2022} =
\frac{\text{unemployed\_2022}}{\text{civilian\_labor\_force\_2022}}
$$

Quality checks:
- Labor force must be positive
- Unemployment rate must lie in [0, 1]

**Output columns:**
- `fips`
- `county_name`
- labor force counts
- `unemployment_rate_2022`


In [13]:
def preprocess_employment(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_county_fips(df)

    keep = ["fips", "NAME", "B23025_003E", "B23025_004E", "B23025_005E"]
    out = df[keep].copy()
    out = to_numeric_safe(out, ["B23025_003E", "B23025_004E", "B23025_005E"])

    out = out.rename(columns={
        "NAME": "county_name",
        "B23025_003E": "civilian_labor_force_2022",
        "B23025_004E": "employed_2022",
        "B23025_005E": "unemployed_2022",
    })

    out = out.dropna(subset=["civilian_labor_force_2022", "unemployed_2022"]).copy()
    out = out[out["civilian_labor_force_2022"] > 0].copy()

    out["unemployment_rate_2022"] = out["unemployed_2022"] / out["civilian_labor_force_2022"]
    out = out[out["unemployment_rate_2022"].between(0, 1)].copy()

    return out

emp_raw = load_acs_table("employment")
employment = preprocess_employment(emp_raw)
assert_no_pr(employment, "employment")

employment.shape, employment.head()


((3144, 6),
     fips              county_name  civilian_labor_force_2022  employed_2022  unemployed_2022  unemployment_rate_2022
 1  01001  Autauga County, Alabama                      27163          26411              752                0.027685
 2  01003  Baldwin County, Alabama                     111079         107254             3825                0.034435
 3  01005  Barbour County, Alabama                       8968           8452              516                0.057538
 4  01007     Bibb County, Alabama                       9028           8242              786                0.087062
 5  01009   Blount County, Alabama                      26434          24856             1578                0.059696)

In [14]:
emp_out = PROCESSED_DIR / ACS_TABLES["employment"]["output"]
employment.to_csv(emp_out, index=False)
save_processed(employment, "employment")


Saved: data/processed/acs_employment_b23025_2022.csv


## Building the Final ACS Feature Table

All processed tables are combined into a single county-level dataset.

### 1. Uniqueness checks
Each table is verified to contain **exactly one row per county**. Duplicate FIPS codes raise an explicit error.

### 2. County name handling
To avoid column conflicts, `county_name` is retained only from the income table and dropped from others prior to merging.

### 3. Inner-join merge
Tables are merged sequentially using **inner joins on `fips`**:


In [15]:
def assert_unique_fips(df: pd.DataFrame, name: str) -> None:
    n_unique = df["fips"].nunique()
    if n_unique != len(df):
        raise ValueError(f"[{name}] FIPS is not unique: {n_unique} unique vs {len(df)} rows")

def drop_non_key_name(df: pd.DataFrame) -> pd.DataFrame:
    
    cols = [c for c in df.columns if c != "county_name"]
    return df[cols].copy()

# 1) uniqueness checks
for name, df_ in [
    ("income", income),
    ("population", population),
    ("education", education),
    ("household", household),
    ("employment", employment),
]:
    assert_unique_fips(df_, name)

# 2) keep county_name only from income
population_m = drop_non_key_name(population)
education_m  = drop_non_key_name(education)
household_m  = drop_non_key_name(household)
employment_m = drop_non_key_name(employment)

# 3) merge
acs_features = income.merge(population_m, on="fips", how="inner")
acs_features = acs_features.merge(education_m,  on="fips", how="inner")
acs_features = acs_features.merge(household_m,  on="fips", how="inner")
acs_features = acs_features.merge(employment_m, on="fips", how="inner")

assert_no_pr(acs_features, "acs_features")

acs_features.shape, acs_features.head()


((3143, 11),
     fips              county_name  median_income_2022  population_2022  ba_plus_share_2022  hs_plus_share_2022  avg_household_size_2022  \
 0  01001  Autauga County, Alabama             68315.0            58761            0.295586            0.904026                     2.61   
 1  01003  Baldwin County, Alabama             71039.0           233420            0.325616            0.915993                     2.53   
 2  01005  Barbour County, Alabama             39712.0            24877            0.118812            0.764922                     2.43   
 3  01007     Bibb County, Alabama             50669.0            22251            0.109199            0.788634                     2.89   
 4  01009   Blount County, Alabama             57440.0            59077            0.147414            0.817552                     2.70   
 
    civilian_labor_force_2022  employed_2022  unemployed_2022  unemployment_rate_2022  
 0                      27163          26411             

In [16]:
merged_name = "acs_features_2022_county.csv"
acs_features.to_csv(PROCESSED_DIR / merged_name, index=False)
print(f"Saved: data/processed/{merged_name}")


Saved: data/processed/acs_features_2022_county.csv


In [17]:
acs_features.isna().sum().sort_values(ascending=False).head(20)


fips                         0
county_name                  0
median_income_2022           0
population_2022              0
ba_plus_share_2022           0
hs_plus_share_2022           0
avg_household_size_2022      0
civilian_labor_force_2022    0
employed_2022                0
unemployed_2022              0
unemployment_rate_2022       0
dtype: int64

In [18]:
acs_features.describe(include="all").T.head(30)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
fips,3143,3143,56045,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
county_name,3143,3143,"Weston County, Wyoming",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
median_income_2022,3143.0,NaN,NaN,NaN,63311.910595,16820.059834,17900.0,52553.5,60938.0,70605.5,170463.0
population_2022,3143.0,NaN,NaN,NaN,105344.415208,333840.267444,50.0,10840.5,25806.0,68094.5,9936690.0
ba_plus_share_2022,3143.0,NaN,NaN,NaN,0.234985,0.100526,0.045903,0.1653,0.209456,0.278743,0.788715
hs_plus_share_2022,3143.0,NaN,NaN,NaN,0.883515,0.057173,0.333333,0.852752,0.895451,0.925319,0.996904
avg_household_size_2022,3143.0,NaN,NaN,NaN,2.511683,0.253118,1.13,2.36,2.48,2.62,4.99
civilian_labor_force_2022,3143.0,NaN,NaN,NaN,53406.671333,175513.489131,36.0,4719.0,11589.0,32424.0,5235164.0
employed_2022,3143.0,NaN,NaN,NaN,50560.981546,164806.38956,36.0,4471.0,10973.0,30788.5,4869620.0
unemployed_2022,3143.0,NaN,NaN,NaN,2845.689787,10952.799958,0.0,219.0,574.0,1686.0,365544.0


## Summary

This notebook constructs a **clean, consistent, and fully populated ACS county-level feature dataset** for 2022 by:

- Standardizing geographic identifiers
- Enforcing strict data quality and validity rules
- Explicitly documenting and validating data exclusions
- Producing a merge-ready feature matrix aligned with Zillow housing data

The result is a transparent and reliable foundation for downstream exploratory analysis and interpretable regression modeling of U.S. county-level housing prices.
